In [1]:
import json
import os
import pandas as pd
from openai import OpenAI
import openai
import numpy as np

In [ ]:

# pip install langchain_openai
from langchain_openai import OpenAIEmbeddings

embeddings_1024 = OpenAIEmbeddings(model="text-embedding-3-large", dimensions=1024)


get JEL codes

# Create Embeddings for JEL codes' description

In [3]:
# # load AEA_Information.csv
# jel_full = pd.read_csv('AEA_Information.csv')

# # Pad the 'JEL Code' column with zeros to ensure it has at least 3 characters
# jel_full['JEL Code'] = jel_full['JEL Code'].str.pad(width=3, side='right', fillchar='0')

# # rename JEL Code to code
# jel_full = jel_full.rename(columns={'JEL Code': 'code'})


In [ ]:
# # Load the JSON data from file
# file_path = 'jel_descriptions.json'
# with open(file_path, 'r', encoding='utf-8') as file:
#     json_data = json.load(file)

# # Convert the JSON data into a DataFrame
# df_codes = pd.DataFrame(list(json_data.items()), columns=['code', 'description'])

# # Function to generate the concatenated description
# def concatenate_description(code, df):
#     level_1_code = code[0] + "00"
#     level_2_code = code[:2] + "0"
    
#     desc_level_1 = df[df['code'] == level_1_code]['description'].values[0] if level_1_code in df['code'].values else ""
#     desc_level_2 = df[df['code'] == level_2_code]['description'].values[0] if level_2_code in df['code'].values else ""
    
#     concatenated_desc = f"{desc_level_1}; {desc_level_2}; {df[df['code'] == code]['description'].values[0]}"
#     return concatenated_desc.strip('; ')

# # Apply the function to generate the full descriptions
# df_codes['description'] = df_codes['code'].apply(lambda x: concatenate_description(x, df_codes))

# # Display the head of the DataFrame to ensure the concatenation worked
# print(df_codes.head())


combine JEL codes' description and title

In [ ]:

# # Step 3: Merge the two datasets on the 'code' column
# df_combined = pd.merge(df_codes, jel_full, on='code', how='left')

# # Step 4: Clean up 'guideline' and 'keywords' columns
# # Replace 'Guideline: Not Specified' and 'Keywords: None Specified' with empty strings
# df_combined['guideline'] = df_combined['guideline'].replace("Guideline: Not Specified", "")
# df_combined['keywords'] = df_combined['keywords'].replace("Keywords: None Specified", "")

# # Handle NaN values by replacing them with empty strings
# df_combined['guideline'] = df_combined['guideline'].fillna("")
# df_combined['keywords'] = df_combined['keywords'].fillna("")

# # Step 5: Function to concatenate description, guideline, and keywords
# def concatenate_description(row):
#     parts = [row['description'], row['guideline'], row['keywords']]
#     # Filter out empty strings and join non-empty parts with '; '
#     return '; '.join(part for part in parts if part).strip('; ')

# # Apply the function to generate the new combined description
# df_combined['description'] = df_combined.apply(concatenate_description, axis=1)

# # Step 6: Drop unnecessary columns (if you want to keep only 'code' and 'description')
# df_combined = df_combined[['code', 'description']]

# # save df_combined as csv
# df_combined.to_csv('jel_combined.csv', index=False)

In [ ]:

# # Step 7: Generate embeddings for the descriptions
# embedded_strings = [f"{desc}" for desc in df_combined['description']]
# ls_embedded = embeddings_1024.embed_documents(embedded_strings)

# # Attach embeddings back to the DataFrame
# df_embeddings = pd.DataFrame({
#     'code': df_combined['code'],
#     'description': df_combined['description'],
#     'embedding': ls_embedded
# })

# # Check if there are duplicate embeddings
# print(df_embeddings['embedding'].duplicated().sum())

# # Save the DataFrame with embeddings to disk
# df_embeddings.to_pickle('df_embedding_codes.pkl')

# # Optionally, delete intermediate variables to free up memory
# del df_combined, ls_embedded, embedded_strings

# # Display the head of the final DataFrame to verify the result
# print(df_embeddings.head())


In [ ]:

# # Verify saved DataFrame
# df_embeddings = pd.read_pickle('df_embedding_codes.pkl')
# print(df_embeddings.head())


# 1. full text: Create Embeddings for source, sink and DAG

In [3]:
import json
import os
import pandas as pd
import openai
import numpy as np
from langchain_openai import OpenAIEmbeddings
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------------------------------------------
# 1. LOAD YOUR MAIN DATA
# -----------------------------------------------------------------
df_full_meta = pd.read_csv("extracted_edges_titles_abstracts_top100.csv")

# (Optional) filter out rows where source/sink are NaN
print(f"Total rows before filtering: {len(df_full_meta)}")
df_full_meta.dropna(subset=['source'], inplace=True)
print(f"Total rows after filtering 'source': {len(df_full_meta)}")
df_full_meta.dropna(subset=['sink'], inplace=True)
print(f"Total rows after filtering 'sink': {len(df_full_meta)}")


Total rows before filtering: 1116344
Total rows after filtering 'source': 1091270
Total rows after filtering 'sink': 1090818


In [4]:
# -----------------------------------------------------------------
# 2. PREPARE YOUR JEL EMBEDDINGS
# -----------------------------------------------------------------
# Load the JEL embeddings DataFrame
df_jel = pd.read_pickle("../3_causal_claims/v7/df_embedding_codes.pkl")
# df_jel should have columns: ['code', 'description', 'embedding']
# Convert embeddings to a NumPy 2D array: shape (#jel_terms, embedding_dim)
jel_embeddings = np.vstack(df_jel['embedding'].to_numpy())

# -----------------------------------------------------------------
# 3. DEFINE YOUR get_top_matches FUNCTION
# -----------------------------------------------------------------
def get_top_matches(cosine_sim_matrix, df_jel, column_name):
    """
    For each embedding in the batch (#items_in_batch), find the
    row in df_jel with the highest similarity (argmax across #jel_terms).
    """
    indices = np.argmax(cosine_sim_matrix, axis=0)
    # For each item in the batch, pick the JEL row (code, description) with the highest similarity
    matches = df_jel.iloc[indices][['code', 'description']].reset_index(drop=True)
    matches.columns = [f'jel_{column_name}', f'jel_{column_name}_description']
    return matches


In [5]:

# -----------------------------------------------------------------
# 4. CHUNK / BATCH PROCESSING
# -----------------------------------------------------------------
# Example: choose a batch size
BATCH_SIZE = 10000

num_rows = len(df_full_meta)
num_batches = (num_rows // BATCH_SIZE) + (1 if num_rows % BATCH_SIZE != 0 else 0)
print(f"Total rows: {num_rows} -> Processing in {num_batches} batch(es).")

# Create a directory for storing temporary batch results
os.makedirs("temp_batch_results_jel", exist_ok=True)
batch_result_paths = []

# (Optional) if you have an embeddings object: embeddings_1024
# from your code: embeddings_1024 = ...
# make sure it is defined before the loop

for i in range(num_batches):
    start_idx = i * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, num_rows)

    print(f"Processing batch {i+1}/{num_batches} with rows {start_idx} to {end_idx-1}...")

    # Slice the DataFrame for this batch
    df_batch = df_full_meta.iloc[start_idx:end_idx].copy()

    # -----------------------------------------------------------------
    # 5. Generate embeddings for 'source' and 'sink'
    # -----------------------------------------------------------------
    # Convert columns to string (avoid errors if any are numeric)
    source_strings = df_batch['source'].astype(str).tolist()
    sink_strings = df_batch['sink'].astype(str).tolist()

    # NOTE: Ensure you have an object like embeddings_1024 to call:
    #       embeddings_1024.embed_documents(list_of_strings)
    #       Replace with your actual embedding function/variable name.
    source_embedded = embeddings_1024.embed_documents(source_strings)
    sink_embedded = embeddings_1024.embed_documents(sink_strings)

    # Convert to NumPy arrays for use in cosine_similarity
    source_embeddings = np.vstack(source_embedded)  # shape: (#items_in_batch, dim)
    sink_embeddings = np.vstack(sink_embedded)      # shape: (#items_in_batch, dim)

    # -----------------------------------------------------------------
    # 6. Compute Cosine Similarity
    # -----------------------------------------------------------------
    # jel_embeddings shape: (#jel_terms, dim)
    # source_embeddings shape: (#items_in_batch, dim)
    # => cosine_similarity -> shape: (#jel_terms, #items_in_batch)
    cosine_sim_source = cosine_similarity(jel_embeddings, source_embeddings)
    cosine_sim_sink = cosine_similarity(jel_embeddings, sink_embeddings)

    # -----------------------------------------------------------------
    # 7. Find Top Matches
    # -----------------------------------------------------------------
    matches_source = get_top_matches(cosine_sim_source, df_jel, "source")
    matches_sink = get_top_matches(cosine_sim_sink, df_jel, "sink")

    # -----------------------------------------------------------------
    # 8. Concatenate the JEL code columns onto our batch DataFrame
    # -----------------------------------------------------------------
    df_batch.reset_index(drop=True, inplace=True)
    df_batch = pd.concat([df_batch, matches_source, matches_sink], axis=1)

    # -----------------------------------------------------------------
    # 9. Write partial batch results to a temporary file
    # -----------------------------------------------------------------
    batch_path = f"temp_batch_results_jel/batch_{i}.parquet"
    df_batch.to_parquet(batch_path, engine="pyarrow")
    batch_result_paths.append(batch_path)

    # Cleanup
    del df_batch, source_embedded, sink_embedded
    del source_embeddings, sink_embeddings, cosine_sim_source, cosine_sim_sink
    del matches_source, matches_sink

Total rows: 1090818 -> Processing in 110 batch(es).
Processing batch 1/110 with rows 0 to 9999...


c:\Users\pgarg1\AppData\Local\anaconda3\Lib\site-packages\pyarrow\pandas_compat.py:373: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if _pandas_api.is_sparse(col):


Processing batch 2/110 with rows 10000 to 19999...


c:\Users\pgarg1\AppData\Local\anaconda3\Lib\site-packages\pyarrow\pandas_compat.py:373: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if _pandas_api.is_sparse(col):


Processing batch 3/110 with rows 20000 to 29999...


...

c:\Users\pgarg1\AppData\Local\anaconda3\Lib\site-packages\pyarrow\pandas_compat.py:373: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if _pandas_api.is_sparse(col):


Processing batch 109/110 with rows 1080000 to 1089999...


c:\Users\pgarg1\AppData\Local\anaconda3\Lib\site-packages\pyarrow\pandas_compat.py:373: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if _pandas_api.is_sparse(col):


Processing batch 110/110 with rows 1090000 to 1090817...


c:\Users\pgarg1\AppData\Local\anaconda3\Lib\site-packages\pyarrow\pandas_compat.py:373: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if _pandas_api.is_sparse(col):


In [6]:
# -----------------------------------------------------------------
# 10. AFTER ALL BATCHES, COMBINE RESULTS
# -----------------------------------------------------------------
df_all = pd.concat([pd.read_parquet(p) for p in batch_result_paths], ignore_index=True)

# Save the final combined DataFrame
df_all.to_parquet("df_full_meta_with_jel_top100.parquet", engine="pyarrow")

# (Optional) remove temporary files
for p in batch_result_paths:
    os.remove(p)

print("Final DataFrame with JEL codes (sample rows):")
print(df_all.head(10))

c:\Users\pgarg1\AppData\Local\anaconda3\Lib\site-packages\pyarrow\pandas_compat.py:373: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if _pandas_api.is_sparse(col):


Final DataFrame with JEL codes (sample rows):
      paper_id  edge_number  paper_edge_id  \
0  W1491339034            1  W1491339034_1   
1  W1491339034            2  W1491339034_2   
2  W1491339034            3  W1491339034_3   
3  W1491339034            4  W1491339034_4   
4  W1491339034            5  W1491339034_5   
5  W1491339034            6  W1491339034_6   
6  W1481328804            1  W1481328804_1   
7  W2620990083            1  W2620990083_1   
8  W2991892933            1  W2991892933_1   
9  W2991892933            2  W2991892933_2   

                                               claim  \
0            recycling incentives -> recycling rates   
1  Federal government incentives -> competitive p...   
2            market failure -> environmental damages   
3         resource recovery -> environmental damages   
4  Federal regulation -> secondary material recovery   
5               municipal subsidization -> recycling   
6                                D* -> real net debt   

In [7]:
df_all

,paper_id,edge_number,paper_edge_id,claim,source,sink,is_causal_relationship,justification,sign_of_impact,effect_size,...,temporal_context,start_year,end_year,geographical_context_countries,ownership,ownership_details,jel_source,jel_source_description,jel_sink,jel_sink_description
0,W1491339034,1,W1491339034_1,recycling incentives -> recycling rates,recycling incentives,recycling rates,True,The paper discusses that proposals for recycli...,increase,None,...,None,[],[],[],[],[],L99,Industrial Organization: General; Industry Stu...,L99,Industrial Organization: General; Industry Stu...
1,W1491339034,2,W1491339034_2,Federal government incentives -> competitive p...,Federal government incentives,competitive position of secondary materials se...,True,The abstract states that the Federal governmen...,increase,None,...,None,[],[],[],[],[],H81,Public Economics: General; Public Economics: M...,L99,Industrial Organization: General; Industry Stu...
2,W1491339034,3,W1491339034_3,market failure -> environmental damages,market failure,environmental damages,True,The paper discusses that market failure attrib...,increase,None,...,None,[],[],[],[],[],D52,Microeconomics: General; General Equilibrium a...,Q53,Agricultural and Natural Resource Economics; E...
3,W1491339034,4,W1491339034_4,resource recovery -> environmental damages,resource recovery,environmental damages,True,The paper mentions that resource recovery woul...,decrease,None,...,None,[],[],[],[],[],L99,Industrial Organization: General; Industry Stu...,Q53,Agricultural and Natural Resource Economics; E...
4,W1491339034,5,W1491339034_5,Federal regulation -> secondary material recovery,Federal regulation,secondary material recovery,True,The abstract states that the existing structur...,decrease,None,...,None,[],[],[],[],[],L51,Industrial Organization: General; Regulation a...,L99,Industrial Organization: General; Industry Stu...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1090813,W3098438983,2,W3098438983_2,preventative and protective measures against C...,preventative and protective measures against C...,food and plastic waste generation,True,The paper evaluates the implications of the me...,increase,None,...,"immediate, short-term and medium-term perspective",[],[],[],[],[],D18,Microeconomics: General; Household Behavior: G...,L99,Industrial Organization: General; Industry Stu...
1090814,W3098438983,3,W3098438983_3,hospitality sector -> integration into alterna...,hospitality sector,integration into alternative food networks (AF...,True,"To address the issue of food waste, the hospit...",increase,None,...,None,[],[],[],[],[],Z31,Other Special Topics: General; Tourism Economi...,Q13,Agricultural and Natural Resource Economics; E...
1090815,W3098438983,4,W3098438983_4,investment in 'green' innovation -> reduction ...,investment in 'green' innovation,reduction of plastic waste,True,The hospitality sector should invest in 'green...,decrease,None,...,None,[],[],[],[],[],Q55,Agricultural and Natural Resource Economics; E...,L99,Industrial Organization: General; Industry Stu...
1090816,W3098438983,5,W3098438983_5,targeted policy interventions -> support for g...,targeted policy interventions,support for green innovation,True,Investment in 'green' innovation needs to be e...,increase,None,...,None,[],[],[],[],[],J68,Labor and Demographic Economics: General; Mobi...,Q55,Agricultural and Natural Resource Economics; E...
